<div dir="rtl">

# 02 — ترميز DDXPlus إلى مصفوفة ميزات (Baseline Encoding)

هذا النوتبوك يبني **خط أساس** لتمثيل الأدلة (EVIDENCES) والخصائص الديموغرافية كمصفوفة أرقام صالحة
لتغذية نموذج ML، بالاعتماد حصراً على تعريفات `release_evidences.json` (من `01_explore.ipynb`).

**خارج النطاق عمداً في هذا النوتبوك:**
- لا ترميز ثلاثي (مثل تمييز "لم يُسأل" عن "سُئل ونفى").
- لا dropout / إخفاء عشوائي للميزات.
- لا تدريب نموذج — تلك خطوة منفصلة لاحقة، لم تُفتح بعد.

</div>

In [1]:
import ast
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

<div dir="rtl">

## بناء فهرس الأعمدة من `release_evidences.json` حصراً

قواعد الترميز حسب `data_type` (مأخوذة من الملف نفسه، دون أي مصدر آخر):

- **B (ثنائي، 208 دليلاً)**: عمود واحد لكل دليل — 1 إن ظهر الكود في `EVIDENCES`، وإلا 0.
- **C (فئوي أحادي الاختيار، 10 أدلة)**: **one-hot** — عمود لكل قيمة ممكنة (88 عموداً إجمالاً)، عمود واحد فقط
  نشط لكل مريض ضمن نفس الدليل.
- **M (متعدد الاختيار، 5 أدلة)**: **multi-hot** — عمود لكل قيمة ممكنة (676 عموداً إجمالاً)، يمكن أن يتفعّل أكثر
  من عمود لنفس الدليل عند نفس المريض (مثال: مواقع الألم المتعددة).

زائد `AGE` (قيمة رقمية خام، غير مرمَّزة) و`SEX_M` (1 إذا `SEX == 'M'`، وإلا 0 — القيمتان F/M فقط يُتحقَّق منهما أدناه).

</div>

In [2]:
with open(DATA_RAW / "release_evidences.json", encoding="utf-8") as f:
    evidences = json.load(f)


def build_feature_index(evidences: dict) -> tuple[list[str], dict[str, int]]:
    columns = []
    for code in sorted(evidences.keys(), key=lambda c: int(c.split("_")[1])):
        v = evidences[code]
        if v["data_type"] == "B":
            columns.append(code)
        else:  # 'C' (one-hot) أو 'M' (multi-hot): عمود لكل قيمة ممكنة
            for value in v["possible-values"]:
                columns.append(f"{code}_@_{value}")
    item_to_col_idx = {name: idx for idx, name in enumerate(columns)}
    return columns, item_to_col_idx


evidence_columns, item_to_col_idx = build_feature_index(evidences)
feature_names = evidence_columns + ["AGE", "SEX_M"]

n_binary = sum(1 for v in evidences.values() if v["data_type"] == "B")
n_cat = sum(len(v["possible-values"]) for v in evidences.values() if v["data_type"] == "C")
n_multi = sum(len(v["possible-values"]) for v in evidences.values() if v["data_type"] == "M")

print(f"أعمدة ثنائية (B): {n_binary}")
print(f"أعمدة فئوية one-hot (C): {n_cat}")
print(f"أعمدة متعددة القيم multi-hot (M): {n_multi}")
print(f"إجمالي أعمدة الأدلة: {len(evidence_columns)}")
print(f"إجمالي الأعمدة (+ AGE + SEX_M): {len(feature_names)}")

أعمدة ثنائية (B): 208
أعمدة فئوية one-hot (C): 88
أعمدة متعددة القيم multi-hot (M): 676
إجمالي أعمدة الأدلة: 972
إجمالي الأعمدة (+ AGE + SEX_M): 974


<div dir="rtl">

## دالة الترميز: DataFrame → مصفوفة sparse

ترميز ثنائي عادي (1 موجود / 0 غير موجود) لكل عنصر في `EVIDENCES` عبر الفهرس أعلاه، زائد عمود `AGE` الرقمي
الخام وعمود `SEX_M`. أي عنصر في `EVIDENCES` لا يطابق عموداً معروفاً يُسقِط الترميز فوراً (`assert`) بدل تجاهله
بصمت.

</div>

In [3]:
def encode_evidences(evidences_parsed: pd.Series, item_to_col_idx: dict, n_evidence_cols: int) -> sparse.csr_matrix:
    exploded = evidences_parsed.explode()
    col_idx_series = exploded.map(item_to_col_idx)
    assert col_idx_series.notna().all(), "عناصر EVIDENCES لا تطابق أي عمود في الفهرس"

    col_idx = col_idx_series.to_numpy(dtype=np.int64)
    row_idx = exploded.index.to_numpy()
    data = np.ones(len(exploded), dtype=np.int8)

    return sparse.coo_matrix(
        (data, (row_idx, col_idx)), shape=(len(evidences_parsed), n_evidence_cols)
    ).tocsr()


def encode_split(df: pd.DataFrame) -> sparse.csr_matrix:
    assert set(df["SEX"].unique()) <= {"F", "M"}, "قيم SEX غير متوقعة خارج {F, M}"

    evidence_matrix = encode_evidences(df["EVIDENCES_PARSED"], item_to_col_idx, len(evidence_columns))
    age_col = sparse.csr_matrix(df["AGE"].to_numpy().reshape(-1, 1))
    sex_col = sparse.csr_matrix((df["SEX"] == "M").astype(np.int8).to_numpy().reshape(-1, 1))

    return sparse.hstack([evidence_matrix, age_col, sex_col]).tocsr()

<div dir="rtl">

## تحميل الانقسامات الثلاثة وترميزها

</div>

In [4]:
SPLIT_FILES = {
    "train": "release_train_patients.csv",
    "validate": "release_validate_patients.csv",
    "test": "release_test_patients.csv",
}

splits = {}
features = {}

for split_name, filename in SPLIT_FILES.items():
    df = pd.read_csv(DATA_RAW / filename)
    df["EVIDENCES_PARSED"] = df["EVIDENCES"].apply(ast.literal_eval)
    splits[split_name] = df
    features[split_name] = encode_split(df)
    print(f"{split_name}: {df.shape[0]} صف → مصفوفة {features[split_name].shape}")

train: 1025602 صف → مصفوفة (1025602, 974)


validate: 132448 صف → مصفوفة (132448, 974)


test: 134529 صف → مصفوفة (134529, 974)


<div dir="rtl">

## ترميز التسمية (PATHOLOGY) عبر LabelEncoder

يُدرَّب `LabelEncoder` على `train` فقط، ثم يُطبَّق على `validate`/`test` بعد التحقق من عدم وجود أمراض غير معروفة.

</div>

In [5]:
label_encoder = LabelEncoder().fit(splits["train"]["PATHOLOGY"])

for split_name in ("validate", "test"):
    unseen = set(splits[split_name]["PATHOLOGY"]) - set(label_encoder.classes_)
    assert not unseen, f"أمراض غير معروفة في {split_name}: {unseen}"

labels = {
    split_name: label_encoder.transform(df["PATHOLOGY"])
    for split_name, df in splits.items()
}

print(f"عدد الفئات (الأمراض): {len(label_encoder.classes_)}")

عدد الفئات (الأمراض): 49

<div dir="rtl">

## التحقق الصريح 1: تطابق عدد وترتيب الأعمدة عبر التقسيمات الثلاثة

</div>

In [6]:
n_cols = {split_name: mat.shape[1] for split_name, mat in features.items()}
print(n_cols)

assert len(set(n_cols.values())) == 1, "عدد الأعمدة غير متطابق بين التقسيمات!"
assert n_cols["train"] == len(feature_names), "عدد الأعمدة لا يطابق feature_names!"
print(
    f"تطابق تام: {len(feature_names)} عموداً بنفس الترتيب عبر train/validate/test "
    "(الأعمدة الثلاثة استُخدم لها نفس feature_names/item_to_col_idx المبنيّين مرة واحدة من release_evidences.json)."
)

{'train': 974, 'validate': 974, 'test': 974}
تطابق تام: 974 عموداً بنفس الترتيب عبر train/validate/test (الأعمدة الثلاثة استُخدم لها نفس feature_names/item_to_col_idx المبنيّين مرة واحدة من release_evidences.json).


<div dir="rtl">

## التحقق الصريح 2: مجموع قيم كل صف (أعمدة الأدلة فقط) = عدد عناصر EVIDENCES لنفس المريض

أعمدة `AGE`/`SEX_M` مستبعدة من هذا التحقق عمداً — ليست جزءاً من `EVIDENCES`.

</div>

In [7]:
for split_name, df in splits.items():
    evidence_only = features[split_name][:, : len(evidence_columns)]
    row_sums = np.asarray(evidence_only.sum(axis=1)).ravel()
    expected = df["EVIDENCES_PARSED"].apply(len).to_numpy()

    assert np.array_equal(row_sums, expected), f"عدم تطابق مجموع الصف مع عدد عناصر EVIDENCES في {split_name}!"
    print(f"{split_name}: تطابق تام لكل الصفوف الـ{len(df)} (مجموع أعمدة الأدلة == عدد عناصر EVIDENCES).")

train: تطابق تام لكل الصفوف الـ1025602 (مجموع أعمدة الأدلة == عدد عناصر EVIDENCES).
validate: تطابق تام لكل الصفوف الـ132448 (مجموع أعمدة الأدلة == عدد عناصر EVIDENCES).
test: تطابق تام لكل الصفوف الـ134529 (مجموع أعمدة الأدلة == عدد عناصر EVIDENCES).


<div dir="rtl">

## الحفظ: sparse .npz + feature_names.json + label_encoder

</div>

In [8]:
for split_name, mat in features.items():
    sparse.save_npz(DATA_PROCESSED / f"{split_name}_features.npz", mat)
    np.save(DATA_PROCESSED / f"{split_name}_labels.npy", labels[split_name])

with open(DATA_PROCESSED / "feature_names.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

joblib.dump(label_encoder, DATA_PROCESSED / "label_encoder.joblib")

print("تم الحفظ في data/processed/:")
for p in sorted(DATA_PROCESSED.iterdir()):
    print(" -", p.name)

تم الحفظ في data/processed/:
 - feature_names.json
 - label_encoder.joblib
 - test_features.npz
 - test_labels.npy
 - train_features.npz
 - train_labels.npy
 - validate_features.npz
 - validate_labels.npy


<div dir="rtl">

## ملخص وخارج النطاق

- بُني فهرس أعمدة موحَّد (974 عموداً: 208 ثنائي + 88 فئوي one-hot + 676 متعدد multi-hot + `AGE` + `SEX_M`)
  من `release_evidences.json` حصراً، واستُخدم نفسه لترميز train/validate/test — تحقّقنا صراحة من تطابق
  عدد وترتيب الأعمدة عبر الثلاثة، ومن تطابق مجموع كل صف مع طول `EVIDENCES` الخام لنفس المريض.
- **لم يُنفَّذ عمداً في هذا النوتبوك**: ترميز ثلاثي، dropout/إخفاء عشوائي، أو أي تدريب نموذج.
- المخرجات محفوظة في `data/processed/` (`*_features.npz`, `*_labels.npy`, `feature_names.json`,
  `label_encoder.joblib`) لاستخدامها في نوتبوك التدريب القادم — لم يُفتح بعد.

</div>